# 02 — Grid World from Scratch
**Week 3 | RL Fundamentals**

We build a 5×5 Grid World **without any RL library**. This forces you to understand every part of the environment interface that Gymnasium later hides from you.

```
[ S ][ . ][ . ][ . ][ . ]
[ . ][ # ][ . ][ # ][ . ]
[ . ][ . ][ . ][ . ][ . ]
[ . ][ # ][ . ][ # ][ . ]
[ . ][ . ][ . ][ . ][ G ]
```
S = start, G = goal (+10), # = pit (-5 and terminal), . = empty (-0.1 step cost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
np.random.seed(0)

In [ ]:
class GridWorld:
    """
    5x5 Grid World environment.
    Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
    """
    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    ACTION_SYMBOLS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal  = (size-1, size-1)
        self.pits  = {(1,1), (1,3), (3,1), (3,3)}
        self.reset()

    def reset(self):
        self.pos = self.start
        return self._state()

    def _state(self):
        return self.pos[0] * self.size + self.pos[1]   # flat index

    def n_states(self):  return self.size ** 2
    def n_actions(self): return 4

    def step(self, action):
        dr, dc = self.ACTIONS[action]
        r, c = self.pos
        nr = max(0, min(self.size-1, r + dr))
        nc = max(0, min(self.size-1, c + dc))
        self.pos = (nr, nc)

        if self.pos == self.goal:
            return self._state(), +10.0, True
        if self.pos in self.pits:
            return self._state(),  -5.0, True
        return self._state(), -0.1, False

    def render_values(self, V, title='Value Function'):
        grid = np.array(V).reshape(self.size, self.size)
        fig, ax = plt.subplots(figsize=(5, 5))
        im = ax.imshow(grid, cmap='RdYlGn', vmin=grid.min(), vmax=grid.max())
        plt.colorbar(im, ax=ax)
        for r in range(self.size):
            for c in range(self.size):
                marker = ''
                if (r,c) == self.goal:          marker = 'G'
                elif (r,c) == self.start:       marker = 'S'
                elif (r,c) in self.pits:        marker = '✕'
                ax.text(c, r, f'{marker}\n{grid[r,c]:.1f}', ha='center', va='center', fontsize=9)
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout(); plt.show()

    def render_policy(self, policy, title='Policy'):
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.set_xlim(-0.5, self.size-0.5); ax.set_ylim(-0.5, self.size-0.5)
        ax.set_xticks(range(self.size)); ax.set_yticks(range(self.size))
        ax.grid(True, linewidth=0.5)
        for r in range(self.size):
            for c in range(self.size):
                s = r * self.size + c
                if (r,c) == self.goal:    ax.text(c, self.size-1-r, 'G', ha='center', va='center', fontsize=16, color='green')
                elif (r,c) in self.pits: ax.text(c, self.size-1-r, '✕', ha='center', va='center', fontsize=16, color='red')
                else:                    ax.text(c, self.size-1-r, self.ACTION_SYMBOLS[policy[s]], ha='center', va='center', fontsize=18)
        ax.set_title(title); plt.tight_layout(); plt.show()

env = GridWorld()
print(f"States: {env.n_states()}, Actions: {env.n_actions()}")
print(f"Start: {env.start}, Goal: {env.goal}, Pits: {env.pits}")

## 2. Random Policy Rollout

In [ ]:
def rollout(env, policy_fn, max_steps=100):
    state = env.reset()
    total_reward = 0.0
    steps = 0
    done = False
    while not done and steps < max_steps:
        action = policy_fn(state)
        state, reward, done = env.step(action)
        total_reward += reward
        steps += 1
    return total_reward, done, steps

returns = []
for _ in range(5000):
    r, done, _ = rollout(env, lambda s: np.random.randint(4))
    returns.append(r)

plt.figure(figsize=(7, 3))
plt.hist(returns, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Total reward'); plt.ylabel('Episodes')
plt.title(f'Random Policy Returns  (mean={np.mean(returns):.2f})')
plt.tight_layout(); plt.show()

## 3. Visualise a Uniform Random Value Function
(Not yet optimal — we'll fix that in Week 4)

In [ ]:
# Placeholder: uniform random values just to show the visualisation
V_random = np.random.uniform(-5, 10, env.n_states())
# Mark terminal states
V_random[env.goal[0]*env.size + env.goal[1]] = 10
for p in env.pits:
    V_random[p[0]*env.size + p[1]] = -5
env.render_values(V_random, title='Random V (placeholder — will solve in Week 4)')

## ✅ Exercises
1. Add a **stochastic** step function: with probability 0.1, the agent moves in a random direction instead of its intended direction (slippery floor). How does this change the rollout returns?
2. Change the pit penalty from -5 to -1. How does the distribution of returns change?
3. **Challenge**: add a 'treasure' cell at (2,2) that gives +3 reward but does NOT end the episode. Does a random policy find it often?

In [ ]:
10# stochastic version

old_step = env.step

def stochastic_step(action):

    if np.random.rand() < 0.1:
        action = np.random.choice([0,1,2,3])

    return old_step(action)

env.step = stochastic_step


returns = []

for _ in range(100):
    r,_ = rollout(
        env,
        lambda s: np.random.choice(4)
    )
    returns.append(r)

print("Average return:", np.mean(returns))

Observation:
Returns become more variable and usually slightly worse because sometimes the agent moves in unintended directions.

In [ ]:
2) env.pit_reward = -1

returns = []

for _ in range(100):

    r,_ = rollout(
        env,
        lambda s: np.random.choice(4)
    )

    returns.append(r)

plt.hist(returns)

plt.title(
    "Returns after reducing pit penalty"
)

plt.show()

Observation:
Returns become less negative because falling into pits is less costly.
Distribution shifts upward.

3) Treasure:

Position = (2,2)
Reward = +3
Episode should continue

In [ ]:
treasure = (2,2)

count = 0

for _ in range(100):

    state = env.reset()

    done = False

    while not done:

        action = np.random.choice(4)

        state,reward,done = env.step(action)

        if state == treasure:
            count += 1

print(
    "Treasure found:",
    count,
    "times"
)

Observation:
Random policy finds the treasure occasionally but not consistently because actions are random.